# Olist: SARIMA có thắng seasonal naive không?

Đội vận hành Olist muốn dự báo **số đơn mỗi ngày trong 14 ngày tới** để xếp ca kho và đặt xe. Trước
khi dùng một model "xịn", câu hỏi đầu tiên luôn là: nó có thắng baseline ngây thơ nhất không, và
thắng **ổn định** qua nhiều thời điểm dự báo hay chỉ ở một lần chia train/test may mắn?

Chuỗi: số đơn (mọi trạng thái) theo ngày mua `order_purchase_timestamp`, từ thứ Hai 2017-01-09 tới
Chủ nhật 2018-08-19. Trước 2017-01-09 Olist gần như chưa có đơn; sau 2018-08-19 dữ liệu bị cắt dần
(đơn mua cuối tháng 8 chưa vào hết bộ dữ liệu).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật, vẽ, gọi hàm. Test bỏ qua chúng.
- Chấm: `uv run pytest tracks/data-science/modules/07-time-series/exercises/intermediate/02-olist-sarima-vs-naive -q`

In [ ]:
import warnings

import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX

START, END = "2017-01-09", "2018-08-19"

## 1. Dựng chuỗi theo ngày

Đếm đơn theo **ngày** của `order_purchase_timestamp` (bỏ giờ), trong khoảng `[start, end]` tính cả
hai đầu. Ngày không có đơn phải xuất hiện với giá trị 0: một chuỗi thời gian thiếu ngày làm sai mọi
lag, mọi sai phân và mọi chu kỳ mùa vụ.

In [ ]:
def daily_orders(orders: pd.DataFrame, start: str = START, end: str = END) -> pd.Series:
    """Series float, index là mọi ngày từ start tới end (freq D), giá trị là số đơn mua trong ngày."""
    ts = pd.to_datetime(orders["order_purchase_timestamp"])
    counts = ts.dt.normalize().value_counts()
    index = pd.date_range(start, end, freq="D")
    return counts.reindex(index, fill_value=0).astype(float).rename("orders")

In [ ]:
# explore: đọc dữ liệu thật (tải bằng: uv run python data/download_olist.py)
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
orders = pd.read_csv(ROOT / "data/raw/olist_orders_dataset.csv")
y = daily_orders(orders)
print(len(y), "ngày,", int(y.sum()), "đơn; ngày cao nhất:", y.idxmax().date(), int(y.max()))
ax = y.plot(figsize=(11, 3.5), lw=0.8, title="Olist: số đơn mỗi ngày")
y.rolling(28, center=True).mean().plot(ax=ax, lw=2)
plt.show()

## 2. Nhìn cấu trúc chuỗi: STL, stationarity, ACF/PACF

Ba điều cần thấy trước khi chọn model: xu hướng (Olist tăng mạnh suốt 2017), mùa vụ tuần (thứ Bảy
và Chủ nhật ít đơn) và các ngày bất thường (Black Friday 2017-11-24; cuối tháng 5/2018 số đơn giảm
một nửa).

In [ ]:
# explore: STL với chu kỳ 7, robust để Black Friday không làm méo thành phần mùa vụ
from statsmodels.tsa.seasonal import STL

stl = STL(y, period=7, robust=True).fit()
stl.plot()
plt.show()
print(stl.seasonal.groupby(y.index.day_of_week).mean().round(1))

In [ ]:
# explore: ADF (H0: có unit root) và KPSS (H0: stationary) trên chuỗi gốc và chuỗi sai phân
from statsmodels.tsa.stattools import adfuller, kpss

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for name, x in [("gốc", y), ("sai phân bậc 1", y.diff().dropna()), ("sai phân mùa vụ 7", y.diff(7).dropna())]:
        adf_p = adfuller(x, autolag="AIC", result_object=False)[1]
        kpss_p = kpss(x, regression="c", nlags="auto", result_object=False)[1]
        print(f"{name:20s} ADF p = {adf_p:.4f}   KPSS p = {kpss_p:.4f}")

In [ ]:
# explore: ACF/PACF của chuỗi sai phân bậc 1: đỉnh ở lag 7, 14 là dấu hiệu mùa vụ tuần
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
plot_acf(y.diff().dropna(), lags=28, ax=axes[0])
plot_pacf(y.diff().dropna(), lags=28, ax=axes[1])
plt.show()

## 3. Hai forecaster

Cả hai nhận `train` (array theo thời gian, kết thúc ngay trước ngày cần dự báo đầu tiên) và `h`,
trả về `np.ndarray` độ dài `h`.

- `seasonal_naive(train, h, m=7)`: lặp lại tuần cuối cùng.
- `sarima_forecast(train, h)`: `SARIMAX(order=(1, 1, 1), seasonal_order=(0, 1, 1, 7))`, fit với
  `disp=False`, dự báo `h` bước. Đề bài cố định cấu hình này để mọi người so cùng một model; phần
  khám phá ở cuối mời bạn thử cấu hình khác.

In [ ]:
def seasonal_naive(train, h: int, m: int = 7) -> np.ndarray:
    """Lặp lại m giá trị cuối của train cho h bước."""
    y = np.asarray(train, dtype=float)
    return y[-m:][np.arange(h) % m]


def sarima_forecast(train, h: int) -> np.ndarray:
    """SARIMA(1,1,1)(0,1,1)[7] fit trên train, trả dự báo h bước."""
    y = np.asarray(train, dtype=float)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")   # optimizer hay báo chưa hội tụ trên chuỗi có Black Friday
        fit = SARIMAX(y, order=(1, 1, 1), seasonal_order=(0, 1, 1, 7)).fit(disp=False)
    return np.asarray(fit.forecast(h))

## 4. Backtest theo nhiều origin

`backtest_mase(series, forecasters, cutoffs, h=14, m=7)`: với mỗi `cutoff` (một ngày), train là mọi
ngày **trước** cutoff, test là `h` ngày từ cutoff. Mỗi (model, cutoff) cho một dòng với cột
`model, cutoff, mae, mase`. MASE chia MAE cho `mean(|train[t] − train[t − m]|)` tính trên train của
**chính origin đó**.

`summarize(table, baseline="seasonal_naive")`: index là model, cột `mase_mean`, `mase_median`,
`win_rate` (tỉ lệ origin mà model có MASE **nhỏ hơn hẳn** baseline), sắp tăng dần theo `mase_mean`.

In [ ]:
def backtest_mase(series: pd.Series, forecasters: dict, cutoffs, h: int = 14, m: int = 7) -> pd.DataFrame:
    """Một dòng mỗi (model, cutoff): cột model, cutoff, mae, mase."""
    rows = []
    for cutoff in pd.DatetimeIndex(cutoffs):
        train = series[series.index < cutoff].to_numpy(dtype=float)
        in_test = (series.index >= cutoff) & (series.index < cutoff + pd.Timedelta(days=h))
        test = series[in_test].to_numpy(dtype=float)
        scale = np.mean(np.abs(train[m:] - train[:-m]))
        for name, fn in forecasters.items():
            mae = float(np.mean(np.abs(test - fn(train, h))))
            rows.append({"model": name, "cutoff": cutoff, "mae": mae, "mase": mae / scale})
    return pd.DataFrame(rows)


def summarize(table: pd.DataFrame, baseline: str = "seasonal_naive") -> pd.DataFrame:
    """Index model; cột mase_mean, mase_median, win_rate (so với baseline); sắp theo mase_mean."""
    wide = table.pivot(index="cutoff", columns="model", values="mase")
    out = pd.DataFrame({
        "mase_mean": wide.mean(),
        "mase_median": wide.median(),
        "win_rate": wide.lt(wide[baseline], axis=0).mean(),
    })
    return out.sort_values("mase_mean")

In [ ]:
# explore: 12 origin cách nhau 14 ngày, từ 2018-03-04 tới 2018-08-05 (mất khoảng 5–10 giây)
CUTOFFS = pd.date_range("2018-03-04", periods=12, freq="14D")
table = backtest_mase(y, {"seasonal_naive": seasonal_naive, "sarima": sarima_forecast}, CUTOFFS)
display(summarize(table))
table.pivot(index="cutoff", columns="model", values="mase").round(2)

In [ ]:
# explore: nhìn một origin cụ thể, kèm prediction interval 80% của SARIMA
cutoff = pd.Timestamp("2018-05-13")
train = y[y.index < cutoff]
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    fit = SARIMAX(train.to_numpy(), order=(1, 1, 1), seasonal_order=(0, 1, 1, 7)).fit(disp=False)
fc = fit.get_forecast(28)
idx = pd.date_range(cutoff, periods=28, freq="D")
band = fc.conf_int(alpha=0.2)
ax = y["2018-03-01":"2018-07-01"].plot(figsize=(11, 3.5), color="k", lw=1, label="thật")
ax.plot(idx, fc.predicted_mean, label="SARIMA")
ax.fill_between(idx, band[:, 0], band[:, 1], alpha=0.25, label="PI 80%")
ax.plot(idx, seasonal_naive(train, 28), ls="--", label="seasonal naive")
ax.legend()
plt.show()

## 5. Khám phá thêm (không chấm)

- Thử `log1p` trước khi fit, hoặc thêm Black Friday làm biến ngoại sinh (`exog`) cho SARIMAX.
- Thử ETS (`statsmodels.tsa.exponential_smoothing.ets.ETSModel`, trend damped, mùa vụ 7).
- Thử horizon 28 ngày: khoảng cách giữa các model thay đổi thế nào?

## 6. Kết luận cho đội vận hành

Trên 12 origin (14 ngày mỗi lần, tháng 3 tới tháng 8/2018), SARIMA(1,1,1)(0,1,1)[7] có MASE trung
bình 1.19 so với 1.26 của seasonal naive, thắng ở 7/12 origin. Mức cải thiện khoảng 5% là nhỏ so với
độ dao động giữa các origin (MASE từ 0.6 tới 2.9), nên chưa thể nói SARIMA tốt hơn hẳn. Origin
2018-05-13 làm cả hai model sai nặng: số đơn tuần 21–27/5 giảm một nửa, một cú sốc bên ngoài mà
không model nào chỉ nhìn lịch sử đoán được. MASE > 1 ở hầu hết origin nghĩa là dự báo 14 ngày khó
hơn nhiều so với "dự báo tuần tới bằng tuần này" trong lịch sử. Với đội vận hành: dùng SARIMA (hoặc
seasonal naive) cho mức nền, nhưng luôn kèm khoảng dự báo và một kênh để con người điều chỉnh theo
sự kiện đã biết (khuyến mãi, Black Friday, đình công).